# S05 — Backpropagation II

**Week 3 · Wed Sep 9, 2026 · Module 1**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Boyu-Zhang-UOI/dl-f2026-notebooks/blob/main/s05_backpropagation_ii.ipynb)

Every cell below is a worked example from the [S05 reading](https://boyu-zhang-uoi.github.io/dl-f2026/readings/sessions/s05/) — same code, same seeds, same outputs. Run them, then change things and see what breaks: that is what this notebook is for.

Slides for this session: [s05.html](https://boyu-zhang-uoi.github.io/dl-f2026/slides/s05.html)


In [ ]:
# Colab only: install PyTorch if it is missing (local runs already have it).
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("torch") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch"], check=True)
print("environment ready")

## A working engine in sixty lines


*Expected output starts with:* `y      = 0.1974`


In [ ]:
import math

class Value:
    """A scalar that remembers how it was computed, so gradients can flow back."""

    def __init__(self, data, _children=(), _op=""):
        self.data = data          # the scalar value (forward pass)
        self.grad = 0.0           # dL/d(this node), filled in by backward()
        self._backward = lambda: None   # how to push out.grad into the children
        self._prev = set(_children)     # nodes this one was computed from
        self._op = _op            # label for debugging ("+", "*", "tanh")

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), "+")

        def _backward():
            self.grad += out.grad      # d(a+b)/da = 1
            other.grad += out.grad     # d(a+b)/db = 1
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), "*")

        def _backward():
            self.grad += other.data * out.grad   # d(a*b)/da = b
            other.grad += self.data * out.grad   # d(a*b)/db = a
        out._backward = _backward
        return out

    def tanh(self):
        t = math.tanh(self.data)
        out = Value(t, (self,), "tanh")

        def _backward():
            self.grad += (1.0 - t * t) * out.grad   # d tanh(x)/dx = 1 - tanh(x)^2
        out._backward = _backward
        return out

    def backward(self):
        # 1. Topological sort: dependencies/operands before their result.
        topo, visited = [], set()

        def build(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build(child)
                topo.append(v)
        build(self)

        # 2. Seed the output gradient, then walk the graph in reverse.
        self.grad = 1.0
        for node in reversed(topo):
            node._backward()

    __radd__ = __add__
    __rmul__ = __mul__


# A single tanh neuron: y = tanh(w1*x1 + w2*x2 + b)
x1, x2 = Value(1.5), Value(-2.0)
w1, w2 = Value(0.6), Value(0.4)
b = Value(0.1)

s = w1 * x1 + w2 * x2 + b
y = s.tanh()
y.backward()

print(f"y      = {y.data:.4f}")
print(f"dy/dw1 = {w1.grad:.4f}   (should be x1 * (1 - y^2) = {1.5 * (1 - y.data**2):.4f})")
print(f"dy/dw2 = {w2.grad:.4f}")
print(f"dy/db  = {b.grad:.4f}")
print(f"dy/dx1 = {x1.grad:.4f}")

# A value used twice: gradients must ACCUMULATE, not overwrite.
a = Value(3.0)
f = a * a + a          # f = a^2 + a, so df/da = 2a + 1 = 7
f.backward()
print(f"f      = {f.data:.4f}")
print(f"df/da  = {a.grad:.4f}   (should be 2a + 1 = 7)")

## Checking against torch.autograd


*Expected output starts with:* `               micro       torch      |diff|`


In [ ]:
import math
import torch

# --- the same Value class as before, condensed ---
class Value:
    def __init__(self, data, _children=()):
        self.data, self.grad = data, 0.0
        self._backward, self._prev = lambda: None, set(_children)

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other))
        def _backward():
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other))
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out

    def tanh(self):
        t = math.tanh(self.data)
        out = Value(t, (self,))
        def _backward():
            self.grad += (1.0 - t * t) * out.grad
        out._backward = _backward
        return out

    def backward(self):
        topo, visited = [], set()
        def build(v):
            if v not in visited:
                visited.add(v)
                for c in v._prev:
                    build(c)
                topo.append(v)
        build(self)
        self.grad = 1.0
        for node in reversed(topo):
            node._backward()

    __radd__ = __add__
    __rmul__ = __mul__


def expression(a, b, c):
    """Same formula for both engines: reuse, nesting, and a nonlinearity."""
    d = a * b + c
    e = (d + a).tanh()
    return e * d + b


# Ours
a, b, c = Value(0.7), Value(-1.3), Value(2.0)
out = expression(a, b, c)
out.backward()

# PyTorch
at = torch.tensor(0.7, requires_grad=True)
bt = torch.tensor(-1.3, requires_grad=True)
ct = torch.tensor(2.0, requires_grad=True)
outt = expression(at, bt, ct)
outt.backward()

print(f"{'':8s}{'micro':>12s}{'torch':>12s}{'|diff|':>12s}")
print(f"{'out':8s}{out.data:12.6f}{outt.item():12.6f}{abs(out.data - outt.item()):12.2e}")
for name, v, t in [("dL/da", a, at), ("dL/db", b, bt), ("dL/dc", c, ct)]:
    print(f"{name:8s}{v.grad:12.6f}{t.grad.item():12.6f}{abs(v.grad - t.grad.item()):12.2e}")

## Where calculus runs out: kinks and infinities


*Expected output starts with:* `relu'(x) at x = -1, 0, 1 : [0.0, 0.0, 1.0]`


In [ ]:
import torch

torch.manual_seed(0)

# relu is not differentiable at 0. What does autograd return there?
x = torch.tensor([-1.0, 0.0, 1.0], requires_grad=True)
torch.relu(x).sum().backward()
print(f"relu'(x) at x = -1, 0, 1 : {x.grad.tolist()}")

# abs has the same kink at 0
xa = torch.tensor([-1.0, 0.0, 1.0], requires_grad=True)
xa.abs().sum().backward()
print(f"abs'(x)  at x = -1, 0, 1 : {xa.grad.tolist()}")

# sqrt at 0: the one-sided derivative is genuinely infinite
xs = torch.tensor([0.0, 1.0], requires_grad=True)
xs.sqrt().sum().backward()
print(f"sqrt'(x) at x = 0, 1     : {xs.grad.tolist()}")

# The where-trap: masking the VALUE does not mask the GRADIENT.
xw = torch.tensor([0.0, 4.0], requires_grad=True)
safe_looking = torch.where(xw > 0, torch.sqrt(xw), torch.zeros_like(xw))
safe_looking.sum().backward()
print(f"where(x>0, sqrt(x), 0)'  : {xw.grad.tolist()}")

# In-place mutation of a tensor that backward still needs is an error.
a = torch.ones(3, requires_grad=True)
b = a.exp()          # exp's backward rule reuses its OUTPUT b
b.add_(1.0)          # in-place: modifies b after it was saved
try:
    b.sum().backward()
except RuntimeError as e:
    print(f"in-place then backward -> RuntimeError: {str(e)[:75]}...")

## The graph's lifecycle: retention, accumulation, detachment


*Expected output starts with:* `after 1st backward: w.grad = 12.0`


In [ ]:
import torch

torch.manual_seed(0)

# 1. A graph is consumed by backward(); a second call fails.
w = torch.tensor(2.0, requires_grad=True)
loss = (w * w) * 3.0                       # dL/dw = 6w = 12
loss.backward()
print(f"after 1st backward: w.grad = {w.grad.item():.1f}")
try:
    loss.backward()
except RuntimeError as e:
    print(f"2nd backward -> RuntimeError: {str(e)[:60]}...")

# 2. retain_graph=True keeps the buffers; gradients then ACCUMULATE.
w.grad = None
loss = (w * w) * 3.0
loss.backward(retain_graph=True)
loss.backward()
print(f"two backwards with retain_graph: w.grad = {w.grad.item():.1f}  (12 + 12)")

# 3. detach() cuts the recording: b is a constant as far as autograd knows.
a = torch.tensor(3.0, requires_grad=True)
b = (a * a).detach()                       # value 9.0, no history
c = b * a
c.backward()
print(f"c = detach(a*a) * a: dc/da = {a.grad.item():.1f}  (only the last factor counts)")

# 4. no_grad() skips recording entirely: no graph, no backward possible.
with torch.no_grad():
    d = a * a
print(f"inside no_grad: (a*a).requires_grad = {d.requires_grad}")

## Differentiating the derivative


*Expected output starts with:* `f   = tanh(0.5)          = 0.462117`


In [ ]:
import torch

torch.manual_seed(0)

# Higher-order derivatives of tanh at x = 0.5, checked against closed forms.
x = torch.tensor(0.5, requires_grad=True)
y = torch.tanh(x)

(g1,) = torch.autograd.grad(y, x, create_graph=True)   # graph OF the gradient
(g2,) = torch.autograd.grad(g1, x, create_graph=True)
(g3,) = torch.autograd.grad(g2, x)

t = y.item()
print(f"f   = tanh(0.5)          = {t:.6f}")
print(f"f'  autograd = {g1.item():9.6f}   analytic 1 - t^2          = {1 - t*t:9.6f}")
print(f"f'' autograd = {g2.item():9.6f}   analytic -2 t (1 - t^2)   = {-2*t*(1 - t*t):9.6f}")
print(f"f'''autograd = {g3.item():9.6f}   analytic (1 - t^2)(6t^2-2) = {(1 - t*t)*(6*t*t - 2):9.6f}")

# A full Hessian, one call: f(w) = w0^2 * w1 at w = (1.5, -2.0)
def f(w):
    return w[0] ** 2 * w[1]

w = torch.tensor([1.5, -2.0])
H = torch.autograd.functional.hessian(f, w)
print(f"\nHessian of w0^2 * w1 at (1.5, -2.0):")
print(f"  autograd: [[{H[0,0]:.1f}, {H[0,1]:.1f}], [{H[1,0]:.1f}, {H[1,1]:.1f}]]")
print(f"  analytic: [[2*w1, 2*w0], [2*w0, 0]] = [[-4.0, 3.0], [3.0, 0.0]]")

## Try it yourself

1. Extend `Value` with `__pow__` (for constant exponents), `__neg__`, `__sub__`, and `__truediv__` (define `a / b` as `a * b**-1`). Verify each new operation against `torch.autograd` using the comparison harness from the second example.
2. Add `exp()`, then implement `sigmoid` two ways: composed from your primitives (`1 / (1 + (-x).exp())`) and as a single fused operation with local derivative `s * (1 - s)`. Confirm both give identical gradients on several inputs.
3. Deliberately change one `+=` to `=` in `__mul__`. Find the smallest expression whose gradient becomes wrong, and one nontrivial expression whose gradient stays right. Explain both.
4. Using only your `Value` class, fit `y = 2x - 1` from five data points by gradient descent on two parameters `w` and `b` with squared-error loss. You now have end-to-end training with no framework at all.
5. Explain why your `Value` engine cannot compute second derivatives as written (what type does `grad` have, and what would it need to be?). Then sketch — or implement for `+` and `*` only — a version whose `_backward` builds new `Value` graphs instead of accumulating floats, and check `d^2(a*a*a)/da^2 = 6a` against `torch.autograd.grad` with `create_graph=True`.


---

Full discussion of everything above: [S05 reading](https://boyu-zhang-uoi.github.io/dl-f2026/readings/sessions/s05/).
